# Productionizing ML Systems — Masterclass

The gap between "my model works in a notebook" and "my model serves 1000 requests/second reliably." This notebook covers the concepts and tooling that close that gap — the stuff that comes up when an interviewer asks *"how would you deploy this?"* or *"how do you serve this at scale?"*

## What's covered (in execution order)

**Concurrency — the confusing foundation**
1. **The mental model** — where every term in this notebook fits
2. **Concurrency foundations** — I/O-bound vs CPU-bound, the GIL, threading vs multiprocessing vs asyncio, when to use which

**Async programming (the deep dive you asked for)**
3. **Asyncio internals** — the event loop, coroutines, `await`, `gather`, tasks, the cardinal sin of blocking the loop

**Web serving**
4. **FastAPI** — what it is, why it's popular, WSGI vs ASGI, when to use it vs alternatives, building + testing endpoints, Pydantic validation, serving an ML model

**Inference & serving**
5. **Model serving** — latency vs throughput, batching, the serving framework landscape, workers and processes

**Packaging & deployment**
6. **Docker** — images vs containers, writing a Dockerfile for an ML service, layer caching, multi-stage builds
7. **Cloud deployment** — VMs vs containers vs serverless vs managed, the deploy path, autoscaling

**Operating in production**
8. **Observability** — the refined difference between logging, metrics, and tracing; what to measure; ML-specific monitoring (drift)

## A note on what executes vs what's illustrative

Most of this notebook **runs** — the concurrency timing comparisons, asyncio examples, and FastAPI endpoints (via `TestClient`) all execute live so you can see the behavior. A few things are **illustrative only** (clearly marked): Docker builds, cloud deploys, and real network servers can't run inside a notebook. For those, the code is correct and copy-pasteable; you run it in a terminal.

## Stack
- Python 3.11+, asyncio (stdlib), `nest_asyncio` (to make asyncio play nice with Jupyter)
- fastapi, uvicorn, pydantic, httpx
- scikit-learn + joblib (for the model-serving example)


## Setup

**The one Jupyter-specific thing you need to know:** Jupyter already runs an asyncio event loop in the background. That means `asyncio.run()` — the normal way to start async code — fails inside a notebook with `RuntimeError: asyncio.run() cannot be called from a running event loop`.

Two fixes:
1. Use top-level `await` directly (Jupyter supports this).
2. Call `nest_asyncio.apply()` once, which patches the loop to allow nesting — then `asyncio.run()` works too.

We use `nest_asyncio` so the examples work both in Jupyter and as a plain script.

In [ ]:
import time
import asyncio
import threading
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")

import nest_asyncio
nest_asyncio.apply()   # makes asyncio.run() work inside Jupyter's existing event loop

print(f"Python multiprocessing start method: {mp.get_start_method()}")
print(f"CPU cores available: {mp.cpu_count()}")
print("nest_asyncio applied — asyncio.run() will work in this notebook")


---
# 1. The mental model — where everything fits

Before the details, here's the whole landscape so each term has a home. The journey of a prediction from your laptop to production:

```
   [Your notebook]                      [Production service]
   model.predict(X)        ───►         1000 users hitting an API simultaneously

   To get from left to right, you need to answer:

   1. CONCURRENCY: How do I handle many requests at once without them
      waiting in a single-file line?
      → threading / multiprocessing / asyncio

   2. WEB FRAMEWORK: How do requests reach my model over HTTP?
      → FastAPI (+ uvicorn)

   3. SERVING: How do I run inference efficiently — batching, low latency,
      high throughput?
      → the model-serving layer

   4. PACKAGING: How do I ship this so it runs identically everywhere?
      → Docker

   5. DEPLOYMENT: Where does it actually run, and how does it scale?
      → cloud (EC2 / ECS / Cloud Run / Lambda / SageMaker)

   6. OBSERVABILITY: How do I know it's working / find out when it breaks?
      → logging + metrics + tracing
```

Each numbered concern is one section of this notebook. They stack: concurrency is the foundation (it's *why* a web framework can serve many users), the framework wraps your model, serving makes inference efficient, Docker packages it, the cloud runs it, observability watches it.

**The single biggest source of confusion** — and the thing this notebook fixes first — is concurrency: threading vs multiprocessing vs asyncio. Get that straight and the rest follows.

---
# 2. Concurrency foundations

The whole topic of "threading vs multiprocessing vs asyncio" reduces to **two questions**:

1. Is your work **I/O-bound** or **CPU-bound**?
2. Do you need true **parallelism** or just **concurrency**?

Answer those and the choice is mechanical. Let's build up the concepts.

## 2.1 The single most important distinction: I/O-bound vs CPU-bound

This is the insight that makes everything click.

**I/O-bound work** spends most of its time **waiting** — for a network response, a disk read, a database query, an API call. The CPU is idle during the wait. Examples: calling an external API, querying a database, reading files, downloading data.

**CPU-bound work** spends most of its time **computing** — the CPU is pegged at 100%. Examples: matrix multiplication, image resizing, training a model, parsing huge JSON, running inference on a big neural net.

Why it matters:
- For **I/O-bound** work, you don't need more CPUs — you need a way to **not sit idle while waiting**. Start request B while request A is waiting for the network. → **threading or asyncio**.
- For **CPU-bound** work, the CPU is the bottleneck. The only way to go faster is **more CPUs working in parallel**. → **multiprocessing**.

The classic mistake: using threads to speed up CPU-bound work in Python. It doesn't work, because of the GIL.

## 2.2 The GIL (Global Interpreter Lock)

CPython (the standard Python) has a **Global Interpreter Lock**: only **one thread can execute Python bytecode at a time**, even on a multi-core machine. The lock is passed around between threads.

Consequences:
- **CPU-bound + threads = no speedup.** Two threads doing math don't run in parallel — they take turns holding the GIL. You get the overhead of threading with none of the benefit.
- **I/O-bound + threads = big speedup.** When a thread is *waiting* for I/O (network, disk), it **releases the GIL**, letting other threads run. So threads overlap their waiting time.

This is *the* reason Python concurrency is confusing. In most languages, threads parallelize CPU work. In Python, they don't — you need processes for that.

**(Note: Python 3.13+ has an experimental "free-threaded" build without the GIL, but as of 2026 the GIL is still the default and what you'll encounter in production.)**

## 2.3 Demo: I/O-bound work — sync vs threads vs asyncio

We'll simulate I/O-bound work with `time.sleep()` (synchronous wait) and `asyncio.sleep()` (async wait). Each "task" waits 0.5s, simulating a network call. We run 8 of them.

In [ ]:
# Simulate an I/O-bound task: wait 0.5 seconds (like a network call).
N_TASKS = 8
IO_DELAY = 0.5

def io_task_sync(task_id):
    time.sleep(IO_DELAY)        # simulates waiting for network/disk
    return task_id

# --- Approach 1: Sequential (synchronous) ---
start = time.perf_counter()
results = [io_task_sync(i) for i in range(N_TASKS)]
sync_time = time.perf_counter() - start
print(f"Sequential:       {sync_time:.2f}s   (8 tasks x 0.5s, one after another)")


In [ ]:
# --- Approach 2: Threading ---
# Threads release the GIL while waiting on time.sleep, so they overlap.
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=N_TASKS) as executor:
    results = list(executor.map(io_task_sync, range(N_TASKS)))
thread_time = time.perf_counter() - start
print(f"Threading:        {thread_time:.2f}s   (all 8 wait concurrently)")
print(f"Speedup vs sync:  {sync_time / thread_time:.1f}x")


In [ ]:
# --- Approach 3: Asyncio ---
# Single thread, but the event loop switches between tasks during their awaits.
async def io_task_async(task_id):
    await asyncio.sleep(IO_DELAY)    # non-blocking wait — yields control to the loop
    return task_id

async def run_all():
    return await asyncio.gather(*[io_task_async(i) for i in range(N_TASKS)])

start = time.perf_counter()
results = asyncio.run(run_all())
async_time = time.perf_counter() - start
print(f"Asyncio:          {async_time:.2f}s   (all 8 awaited concurrently on ONE thread)")
print(f"Speedup vs sync:  {sync_time / async_time:.1f}x")


**Read the three results:** sequential takes ~4s (8 × 0.5s). Both threading and asyncio take ~0.5s — they overlap all the waiting. The key insight: **for I/O-bound work, both threading and asyncio give the same ~8x speedup here.** They both let you not-wait-idly. The difference between them is *how* they achieve it (and which scales better), which we'll get to.

## 2.4 Demo: CPU-bound work — why threads fail and processes win

Now the opposite case. A CPU-bound task (summing squares in a tight loop). Watch threading fail to help.

In [ ]:
# A genuinely CPU-bound task — no waiting, just computation.
def cpu_task(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

WORK = 5_000_000
N_CPU_TASKS = 4

# --- Sequential ---
start = time.perf_counter()
results = [cpu_task(WORK) for _ in range(N_CPU_TASKS)]
cpu_sync_time = time.perf_counter() - start
print(f"Sequential:       {cpu_sync_time:.2f}s")


In [ ]:
# --- Threading (expect NO speedup — the GIL serializes CPU work) ---
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=N_CPU_TASKS) as executor:
    results = list(executor.map(cpu_task, [WORK] * N_CPU_TASKS))
cpu_thread_time = time.perf_counter() - start
print(f"Threading:        {cpu_thread_time:.2f}s   (GIL prevents parallelism)")
print(f"Speedup vs sync:  {cpu_sync_time / cpu_thread_time:.2f}x   <- ~1x: threads don't help CPU work!")


In [ ]:
# --- Multiprocessing (each process has its own GIL → can run on its own core) ---
# Each process is a separate Python interpreter with its own GIL.
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=N_CPU_TASKS) as executor:
    results = list(executor.map(cpu_task, [WORK] * N_CPU_TASKS))
cpu_proc_time = time.perf_counter() - start
n_cores = mp.cpu_count()
print(f"Multiprocessing:  {cpu_proc_time:.2f}s")
print(f"Speedup vs sync:  {cpu_sync_time / cpu_proc_time:.2f}x")
print(f"\nThis machine has {n_cores} core(s).")
if n_cores == 1:
    print("With only 1 core there is NO room for parallelism — processes take turns on the")
    print("single core, and process-startup + data-pickling overhead can even make it slower.")
    print("On a typical N-core machine this same code gives close to an Nx speedup (capped at N).")
else:
    print(f"Expect up to ~{n_cores}x here (bounded by core count); overhead eats some of it.")


**Read it:** threading gives ~1x on CPU-bound work (the GIL serializes it) — that holds on *any* machine. Multiprocessing's speedup is **bounded by the number of cores**: on a multi-core box this same code approaches an Nx speedup, but on a 1-core machine (like some cloud sandboxes) there's no parallelism to gain and the process/pickling overhead can even make it slightly slower. The principle is the empirical proof of the GIL: threads can't parallelize Python computation; processes can — *given cores to run on*.

The cost of multiprocessing: each process is a full Python interpreter (heavy memory), and data must be **pickled** to cross process boundaries (serialization overhead). For small fast tasks, that overhead can exceed the benefit — multiprocessing is for genuinely heavy CPU work.

## 2.5 The decision table — the thing to memorize

In [ ]:
decision = pd.DataFrame([
    ["I/O-bound, few tasks (<100)",       "Threading",        "Simple; works with blocking libraries (requests, psycopg2)"],
    ["I/O-bound, many tasks (1000s)",     "Asyncio",          "Scales to huge concurrency cheaply; needs async libraries"],
    ["CPU-bound",                          "Multiprocessing",  "Only way to parallelize Python computation (bypasses GIL)"],
    ["CPU-bound, heavy numeric (numpy)",   "Often just numpy", "numpy releases GIL in C; vectorize before parallelizing"],
    ["Mix of I/O and CPU",                 "Asyncio + process pool", "Async for I/O, offload CPU chunks to a ProcessPoolExecutor"],
    ["Web server handling requests",       "Async framework (FastAPI) + multiple worker processes", "Async per-request I/O concurrency; processes for CPU + cores"],
], columns=["Your situation", "Use", "Why"])
decision


## 2.6 Concurrency vs parallelism — the precise distinction

A common interview question. The crisp version (Rob Pike's framing):

- **Concurrency** is *dealing with* many things at once — structuring your program so multiple tasks can be **in progress** in overlapping time periods. One chef juggling several dishes, switching between them.
- **Parallelism** is *doing* many things at once — multiple tasks **literally executing at the same instant** on different cores. Several chefs each cooking one dish.

Mapping to Python:
- **Asyncio** = concurrency, **no** parallelism (single thread, one task runs at a time, but they interleave).
- **Threading** = concurrency, **no** parallelism for CPU work (GIL), but real overlap for I/O waits.
- **Multiprocessing** = concurrency **and** parallelism (multiple processes on multiple cores).

You can have concurrency without parallelism (asyncio) and parallelism is a *kind* of concurrency. The reason this matters: concurrency is enough for I/O-bound work (you just need to not wait idly); parallelism is required for CPU-bound work (you need more compute happening simultaneously).

## 2.7 Interview Q&A — concurrency

**Q: What's the difference between concurrency and parallelism?**
> Concurrency is *structuring* a program so multiple tasks can be in progress over overlapping time — they take turns. Parallelism is multiple tasks *executing simultaneously* on different cores. Asyncio is concurrent but not parallel (one thread). Multiprocessing is both. Concurrency suffices for I/O-bound work; parallelism is needed for CPU-bound work.

**Q: Why doesn't Python threading speed up CPU-bound code?**
> The GIL (Global Interpreter Lock) allows only one thread to execute Python bytecode at a time. For CPU-bound work, threads just take turns holding the GIL — no parallelism, only overhead. For I/O-bound work threads *do* help, because a thread releases the GIL while waiting on I/O, letting others run.

**Q (trap): When would threading actually help in Python?**
> I/O-bound work: network calls, disk reads, database queries, API requests. While one thread waits on I/O, it releases the GIL and others proceed. Also: calls into C extensions that release the GIL (numpy, some crypto libraries) can parallelize even CPU work via threads — but that's the C code parallelizing, not Python.

**Q: Threading vs asyncio — both handle I/O-bound work. When do you pick which?**
> Threading: simpler mental model, works with existing **blocking** libraries (requests, psycopg2), good for moderate concurrency (dozens to low hundreds of tasks). Asyncio: scales to **thousands** of concurrent tasks cheaply (no per-task thread stack/OS overhead), but requires **async-aware libraries** (httpx, aiohttp, asyncpg) — you can't just `await` a blocking call. Rule of thumb: a few blocking calls → threads; massive I/O concurrency with async libs → asyncio.

**Q (trap): I have a CPU-bound task and used `ThreadPoolExecutor` with 8 workers but it's not faster. Why?**
> The GIL. Threads can't parallelize Python computation. Switch to `ProcessPoolExecutor` (true parallelism across cores) — or, if the work is numeric, vectorize with numpy (which releases the GIL in its C routines) before reaching for processes. Also check: process startup + data pickling overhead might dominate for small tasks; multiprocessing pays off only for genuinely heavy work.

**Q: What's the cost of multiprocessing vs threading?**
> Processes are heavyweight: each is a full Python interpreter (high memory), and data crossing process boundaries must be **pickled** (serialization cost + can't share large objects cheaply). Threads are lightweight and share memory directly. So multiprocessing wins only when the CPU parallelism gain outweighs the process + IPC overhead — i.e. for substantial CPU-bound work, not many tiny tasks.


---
# 3. Asyncio deep dive

You said async is counterintuitive. Here's the model that fixes it. The whole thing rests on one idea: **a single thread that, whenever a task would wait, parks it and runs something else instead.**

## 3.1 The event loop — what it actually is

The **event loop** is a `while True` loop running on **one thread**. It maintains a queue of tasks. Each task runs until it hits an `await` on something that isn't ready yet (a network response, a timer). At that `await`, the task **yields control back to the loop**, saying "wake me when this is ready." The loop then runs the next ready task. When the awaited thing completes, the loop resumes the parked task.

```
Event loop (one thread):
  ┌─────────────────────────────────────────────────┐
  │  while True:                                      │
  │     task = pick a ready task from the queue       │
  │     run it until it hits `await something`        │
  │     if `something` isn't ready:                   │
  │         park the task, register a callback        │
  │         move on to the next ready task            │
  │     when `something` completes:                   │
  │         mark the parked task ready again          │
  └─────────────────────────────────────────────────┘
```

The magic: **while task A waits for the network, task B runs.** No thread sits idle. This is why a single async thread can handle thousands of simultaneous connections — they're nearly all waiting at any given moment, and the one thread weaves between the few that are actually ready.

**Crucial consequence:** if a task does CPU work (no `await`), it **hogs the loop** — no other task runs until it finishes. Async is for I/O concurrency, not CPU parallelism.

## 3.2 Coroutines, async def, await — the syntax

- `async def f(): ...` defines a **coroutine function**. Calling `f()` does NOT run it — it returns a **coroutine object** (a paused computation).
- `await x` means "pause here, let the loop run other things, resume when `x` is done." You can only `await` inside an `async def`.
- To actually run a coroutine: `await` it (from inside async code) or `asyncio.run(coro)` (from sync code, top level).

In [ ]:
# Calling an async function does NOT execute it — it returns a coroutine object.
async def greet(name):
    await asyncio.sleep(0.1)        # simulate I/O
    return f"Hello, {name}"

coro = greet("Samarth")
print(f"Calling greet() returns: {type(coro)}")
print("It hasn't run yet — it's a paused computation.\n")

# To run it, await it (or asyncio.run it).
result = asyncio.run(greet("Samarth"))
print(f"After running: {result}")


## 3.3 The cardinal sin: blocking the event loop

The #1 async bug. If you call a **blocking** function (like `time.sleep()` or a synchronous `requests.get()`) inside a coroutine, you **freeze the entire event loop** — every other task stops, because they all share the one thread.

Demo: the right way (`asyncio.sleep`) vs the wrong way (`time.sleep`) inside async tasks.

In [ ]:
# RIGHT: await asyncio.sleep — yields to the loop, tasks overlap.
async def good_task(i):
    await asyncio.sleep(0.5)     # non-blocking — loop runs others during this
    return i

async def run_good():
    return await asyncio.gather(*[good_task(i) for i in range(5)])

start = time.perf_counter()
asyncio.run(run_good())
good_time = time.perf_counter() - start
print(f"Using await asyncio.sleep:  {good_time:.2f}s   (5 tasks overlap → ~0.5s)")


In [ ]:
# WRONG: time.sleep inside a coroutine — BLOCKS the loop, tasks serialize.
async def bad_task(i):
    time.sleep(0.5)              # BLOCKING — freezes the whole loop!
    return i

async def run_bad():
    return await asyncio.gather(*[bad_task(i) for i in range(5)])

start = time.perf_counter()
asyncio.run(run_bad())
bad_time = time.perf_counter() - start
print(f"Using blocking time.sleep:  {bad_time:.2f}s   (5 tasks serialize → ~2.5s)")
print(f"\nThe blocking version is {bad_time/good_time:.1f}x slower — the loop couldn't switch tasks.")
print("This is THE async bug: a sync/blocking call inside async code kills concurrency.")


**The takeaway:** in async code, every potentially-slow operation must be **awaitable and non-blocking**. That's why async needs special libraries: `httpx`/`aiohttp` instead of `requests`, `asyncpg` instead of `psycopg2`, `aiofiles` instead of plain file I/O. If you must call blocking code, offload it with `await asyncio.to_thread(blocking_fn, args)` (runs it in a thread pool so it doesn't block the loop).

## 3.4 gather vs create_task — running things concurrently

Two ways to run coroutines concurrently:

- **`asyncio.gather(*coros)`** — run several coroutines concurrently, wait for **all**, return results in order. The workhorse.
- **`asyncio.create_task(coro)`** — schedule a coroutine to run **in the background** immediately, returns a `Task` you can await later. Use when you want a task running while you do other things.

In [ ]:
# gather: fan out N concurrent operations, collect all results.
async def fetch(item_id):
    await asyncio.sleep(0.2)              # simulate an API call
    return {"id": item_id, "data": item_id ** 2}

async def fetch_many():
    # All 10 "API calls" happen concurrently.
    results = await asyncio.gather(*[fetch(i) for i in range(10)])
    return results

start = time.perf_counter()
results = asyncio.run(fetch_many())
print(f"Fetched {len(results)} items concurrently in {time.perf_counter()-start:.2f}s (vs 2.0s sequential)")
print(f"Sample results: {results[:3]}")


In [ ]:
# create_task: start work in the background, do other things, then await.
async def background_job(name, delay):
    await asyncio.sleep(delay)
    return f"{name} done"

async def orchestrate():
    # Start two jobs running immediately (they begin NOW, concurrently).
    task1 = asyncio.create_task(background_job("job-A", 0.3))
    task2 = asyncio.create_task(background_job("job-B", 0.5))

    print("  Both tasks are now running in the background...")
    # Do some other (sync, fast) work here if we wanted.
    # Then collect results when we need them.
    r1 = await task1
    r2 = await task2
    return r1, r2

start = time.perf_counter()
print("Starting orchestration:")
results = asyncio.run(orchestrate())
print(f"  Results: {results}")
print(f"  Total time: {time.perf_counter()-start:.2f}s (concurrent → ~0.5s, not 0.8s)")


## 3.5 Offloading blocking work — `asyncio.to_thread`

When you're in async code but must call something blocking (a legacy library, a CPU task), don't call it directly. Offload it to a thread so the event loop keeps running.

In [ ]:
# A blocking function we can't change (pretend it's a third-party library).
def legacy_blocking_call(x):
    time.sleep(0.4)         # blocking I/O inside
    return x * 10

async def use_legacy_safely():
    # asyncio.to_thread runs the blocking call in a thread pool — loop stays free.
    results = await asyncio.gather(*[asyncio.to_thread(legacy_blocking_call, i) for i in range(5)])
    return results

start = time.perf_counter()
results = asyncio.run(use_legacy_safely())
print(f"5 blocking calls offloaded to threads: {time.perf_counter()-start:.2f}s (concurrent → ~0.4s)")
print(f"Results: {results}")
print("\nasyncio.to_thread is the bridge: it lets async code use blocking libraries without freezing the loop.")


## 3.6 A realistic example — concurrent "API calls" with timeout and error handling

A production-shaped async pattern: fetch from several sources concurrently, with per-call timeouts and graceful error handling.

In [ ]:
async def fetch_source(source_id, latency, fail=False):
    '''Simulate fetching from an external source. Some are slow, some fail.'''
    await asyncio.sleep(latency)
    if fail:
        raise ConnectionError(f"source {source_id} unreachable")
    return {"source": source_id, "value": source_id * 100}

async def fetch_with_timeout(source_id, latency, fail, timeout=1.0):
    '''Wrap a fetch with a timeout and convert errors into a result dict.'''
    try:
        result = await asyncio.wait_for(fetch_source(source_id, latency, fail), timeout=timeout)
        return {"source": source_id, "status": "ok", "data": result}
    except asyncio.TimeoutError:
        return {"source": source_id, "status": "timeout", "data": None}
    except ConnectionError as e:
        return {"source": source_id, "status": "error", "data": str(e)}

async def fetch_all_sources():
    configs = [
        (0, 0.2, False),   # fast, ok
        (1, 0.3, False),   # ok
        (2, 1.5, False),   # too slow — will time out at 1.0s
        (3, 0.2, True),    # fails with ConnectionError
        (4, 0.4, False),   # ok
    ]
    return await asyncio.gather(*[fetch_with_timeout(sid, lat, fail) for sid, lat, fail in configs])

start = time.perf_counter()
results = asyncio.run(fetch_all_sources())
elapsed = time.perf_counter() - start

print(f"All sources resolved in {elapsed:.2f}s (concurrent; bounded by the 1.0s timeout):\n")
for r in results:
    print(f"  source {r['source']}: {r['status']:<8s} {r['data'] if r['status']!='ok' else 'OK'}")


**This pattern is the bread and butter of async services:** fan out concurrent I/O, bound each with a timeout, and turn failures into structured results instead of crashing the whole batch. `asyncio.gather` with `return_exceptions=True` is an alternative that collects exceptions instead of raising the first one.

## 3.7 Interview Q&A — asyncio

**Q: What is the event loop?**
> A single-threaded loop that manages and schedules coroutines. It runs a task until the task hits an `await` on something not-yet-ready, parks it, and runs another ready task. When the awaited operation completes, the loop resumes the parked task. It's how one thread juggles thousands of concurrent I/O operations — almost all are waiting at any instant, and the loop weaves between the few that are ready.

**Q: What's the difference between a coroutine and a function?**
> Calling a normal function runs it immediately. Calling a coroutine function (`async def`) returns a **coroutine object** — a paused computation that does nothing until you `await` it or pass it to `asyncio.run()`/`gather()`. Coroutines can suspend themselves at `await` points and resume later; regular functions run start-to-finish.

**Q (trap): I put `time.sleep(5)` in my async endpoint and my whole API froze for all users. Why?**
> `time.sleep` is **blocking** — it doesn't yield to the event loop. Since async runs on one thread, blocking it freezes *every* concurrent task/request. Use `await asyncio.sleep(5)` (non-blocking) instead. For unavoidable blocking calls (legacy libs, CPU work), offload with `await asyncio.to_thread(...)` so the loop stays responsive.

**Q: When should I use asyncio vs threading?**
> Both handle I/O-bound concurrency. Asyncio scales to thousands of concurrent tasks cheaply (no per-task OS thread), but requires async-native libraries (httpx, asyncpg, aiofiles). Threading works with existing blocking libraries and is simpler, but each thread has OS overhead so it caps out at hundreds. High-concurrency network services → asyncio. A handful of blocking calls in an otherwise sync program → threads.

**Q: Can asyncio make CPU-bound code faster?**
> No. Asyncio is single-threaded concurrency for **I/O** — it overlaps *waiting*, not *computing*. A CPU-bound coroutine with no `await` just hogs the loop. For CPU parallelism you need multiprocessing. A common production pattern: an async web server (for I/O concurrency) that offloads CPU-heavy inference to a `ProcessPoolExecutor`.

**Q: What does `asyncio.gather` do?**
> Runs multiple coroutines concurrently, waits for all of them, and returns their results as a list in the order passed (regardless of completion order). It's the standard "fan out N concurrent operations and collect results" primitive. With `return_exceptions=True`, failures are returned as exception objects in the results list instead of propagating immediately.

**Q (trap): Does `await` mean "wait here and block"?**
> No — that's the counterintuitive part. `await` means "pause *this coroutine* and let the event loop run *other* coroutines until the awaited thing is ready." It's the opposite of blocking: it's the explicit point where you *release* control so others can progress. Blocking is when you DON'T yield (like `time.sleep`); `await` is precisely how you avoid blocking.


---
# 4. FastAPI

The web framework you'll use to put a model behind an HTTP endpoint. We'll cover what it is, why it's everywhere, when NOT to use it, and build real endpoints you can test **inside this notebook** using `TestClient` (no server needed).

## 4.1 What FastAPI is, and the WSGI vs ASGI distinction

FastAPI is a Python web framework for building APIs. To understand why it's popular, you need one piece of background: **WSGI vs ASGI**.

- **WSGI** (Web Server Gateway Interface) is the old standard. It's **synchronous**: one request is handled at a time per worker; a request that waits on I/O blocks that worker. Flask and Django (classic) are WSGI.
- **ASGI** (Asynchronous Server Gateway Interface) is the modern standard. It's **async-native**: a single worker can handle many concurrent requests by interleaving them at `await` points (exactly the event-loop model from section 3). FastAPI and Starlette are ASGI.

So FastAPI is "async-first." That matters for I/O-bound endpoints (calling a database, another API, an LLM) — one worker concurrently serves many slow requests instead of blocking on each.

**Why FastAPI is everywhere in ML:**
1. **Async-native** (ASGI) — great for I/O-bound model services that call other services.
2. **Pydantic validation** — request/response schemas are Python type hints; invalid input is auto-rejected with clear errors. Huge for reliability.
3. **Automatic interactive docs** — Swagger UI + OpenAPI spec generated for free at `/docs`.
4. **Fast** — among the fastest Python frameworks (built on Starlette + Pydantic).
5. **Type hints everywhere** — editor autocomplete, fewer bugs.

## 4.2 When to use FastAPI vs alternatives

In [ ]:
framework_choice = pd.DataFrame([
    ["FastAPI",   "Async ML/data APIs, microservices, model serving",
     "Async-native, Pydantic validation, auto docs, fast",
     "Slightly more to learn than Flask; async can confuse"],
    ["Flask",     "Simple sync apps, quick prototypes, traditional web apps",
     "Dead simple, huge ecosystem, battle-tested",
     "Sync (WSGI) by default; manual validation; slower"],
    ["Django",    "Full web apps with DB, admin, auth, templates",
     "Batteries included (ORM, admin, auth)",
     "Heavy for a pure API; overkill for model serving"],
    ["Streamlit / Gradio", "Demos, internal tools, quick model UIs",
     "UI in pure Python, near-zero web code",
     "Not for production APIs; not for high traffic"],
    ["BentoML / Ray Serve / Triton", "Dedicated model serving at scale",
     "Built-in batching, versioning, GPU scheduling",
     "More infra; overkill for a single simple model"],
], columns=["Framework", "Use for", "Strengths", "Weaknesses"])
framework_choice


**The short version:** FastAPI is the default for serving an ML model as an API in 2026. Use Flask for trivial sync apps or if your team already lives in it. Use Streamlit/Gradio for demos, not production. Use a dedicated serving framework (BentoML, Ray Serve, Triton, vLLM) when you need heavy-duty batching, GPU scheduling, or LLM-specific serving.

## 4.3 A minimal FastAPI app — tested in-notebook with TestClient

Normally you'd run a FastAPI app with `uvicorn main:app` in a terminal, which starts a server on a port. In a notebook we can't block on a running server — but FastAPI ships with `TestClient`, which lets you make requests to the app **in-process**, no server or port needed. Perfect for learning and for writing tests.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

# Create the app.
app = FastAPI(title="My First API")

# Define a route: GET / returns a JSON object.
@app.get("/")
def read_root():
    return {"message": "Hello, production world"}

# A route with a path parameter.
@app.get("/items/{item_id}")
def read_item(item_id: int):       # type hint -> FastAPI validates & converts
    return {"item_id": item_id, "squared": item_id ** 2}

# Test it in-process — no server needed.
client = TestClient(app)

print("GET / :")
print(" ", client.get("/").json())

print("\nGET /items/42 :")
print(" ", client.get("/items/42").json())

print("\nGET /items/not_a_number (validation kicks in):")
r = client.get("/items/not_a_number")
print(f"  status code: {r.status_code}")    # 422 Unprocessable Entity
print(f"  error: {r.json()['detail'][0]['msg']}")


**Notice the validation:** `item_id: int` isn't just a hint — FastAPI enforces it. Pass `42` and it works; pass `not_a_number` and you get an automatic **422** error with a clear message. You wrote zero validation code.

## 4.4 Pydantic models — request and response validation

For request **bodies** (POST/PUT), you define a **Pydantic model**: a class with typed fields. FastAPI validates incoming JSON against it, rejects bad data, and gives you a typed Python object.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

# Define the expected request body shape.
class PredictionRequest(BaseModel):
    features: list[float] = Field(..., description="Feature vector for the model")
    model_version: Optional[str] = Field(default="v1", description="Which model to use")

# Define the response shape (documents the API and validates output).
class PredictionResponse(BaseModel):
    prediction: float
    model_version: str
    confidence: float

app2 = FastAPI()

@app2.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    # request is a validated PredictionRequest object — request.features is a list[float].
    fake_pred = sum(request.features)         # stand-in for model.predict()
    return PredictionResponse(
        prediction=fake_pred,
        model_version=request.model_version,
        confidence=0.93,
    )

client2 = TestClient(app2)

# Valid request.
print("Valid POST /predict:")
r = client2.post("/predict", json={"features": [1.0, 2.0, 3.0], "model_version": "v2"})
print(f"  {r.json()}")

# Invalid request — features should be numbers, not strings.
print("\nInvalid POST /predict (string in features):")
r = client2.post("/predict", json={"features": ["a", "b"]})
print(f"  status: {r.status_code}")
print(f"  error: {r.json()['detail'][0]['msg']}")

# Missing required field.
print("\nInvalid POST /predict (missing 'features'):")
r = client2.post("/predict", json={"model_version": "v2"})
print(f"  status: {r.status_code}, error: {r.json()['detail'][0]['msg']}")


**This is FastAPI's killer feature for ML:** your API contract is declared as types. Malformed requests are rejected before they reach your model with clear, structured errors. You never write `if 'features' not in request: return error` boilerplate.

## 4.5 sync vs async endpoints in FastAPI

FastAPI lets you write endpoints as either `def` (sync) or `async def` (async). The distinction matters for performance:

- **`async def` endpoint** — runs **on the event loop**. Use when the endpoint does `await`-able I/O (async DB driver, httpx call to another service, async LLM client). Many requests share one worker concurrently.
- **`def` (sync) endpoint** — FastAPI runs it in a **threadpool** so it doesn't block the loop. Use when your work is blocking (a sync DB driver, `model.predict()` on CPU, a blocking library).

**The trap:** putting blocking code inside an `async def` endpoint. That blocks the event loop and kills concurrency for *all* requests (exactly the section 3.3 sin). If your handler is blocking, either make it a plain `def` (FastAPI threadpools it) or offload with `asyncio.to_thread`.

In [ ]:
import asyncio as _asyncio

app3 = FastAPI()

# Async endpoint — for awaitable I/O.
@app3.get("/async-fetch")
async def async_fetch():
    await _asyncio.sleep(0.05)        # pretend: await an async DB / API call
    return {"type": "async", "note": "ran on the event loop, non-blocking"}

# Sync endpoint — FastAPI runs this in a threadpool automatically.
@app3.get("/sync-compute")
def sync_compute():
    time.sleep(0.05)                  # blocking work — safe here because FastAPI threadpools it
    return {"type": "sync", "note": "ran in a threadpool, didn't block the loop"}

client3 = TestClient(app3)
print("GET /async-fetch :", client3.get("/async-fetch").json())
print("GET /sync-compute:", client3.get("/sync-compute").json())


**The rule of thumb:**
- Endpoint does `await`-able async I/O → `async def`.
- Endpoint does blocking work (sync DB, CPU inference, blocking library) → plain `def` (let FastAPI threadpool it).
- Endpoint does heavy CPU work → plain `def`, and consider offloading to a process pool or a separate inference service (a threadpool doesn't escape the GIL).

## 4.6 Serving a real scikit-learn model

Now the real thing: train a model, save it, load it **once** at startup, and serve predictions. Loading the model once (not per request) is the single most important serving optimization.

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# 1. Train a model (normally done offline, separately from the API).
X, y = make_classification(n_samples=1000, n_features=4, n_informative=3,
                            n_redundant=0, random_state=42)
model = RandomForestClassifier(n_estimators=50, random_state=0).fit(X, y)

# 2. Persist it to disk.
joblib.dump(model, "/tmp/model.joblib")
print("Model trained and saved to /tmp/model.joblib")
print(f"Model expects {model.n_features_in_} features")


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from contextlib import asynccontextmanager

# A holder for app-wide state (the loaded model).
ml_models = {}

# Lifespan handler: load the model ONCE at startup, not per request.
@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup: load the model into memory.
    ml_models["clf"] = joblib.load("/tmp/model.joblib")
    print("[startup] model loaded into memory")
    yield
    # Shutdown: clean up.
    ml_models.clear()
    print("[shutdown] model unloaded")

serving_app = FastAPI(lifespan=lifespan)

class Features(BaseModel):
    features: list[float]

class Prediction(BaseModel):
    predicted_class: int
    probability: float

@serving_app.get("/health")
def health():
    # Health check endpoint — load balancers ping this.
    return {"status": "healthy", "model_loaded": "clf" in ml_models}

@serving_app.post("/predict", response_model=Prediction)
def predict(payload: Features):
    model = ml_models.get("clf")
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    if len(payload.features) != model.n_features_in_:
        raise HTTPException(
            status_code=400,
            detail=f"Expected {model.n_features_in_} features, got {len(payload.features)}",
        )
    X_one = np.array(payload.features).reshape(1, -1)
    proba = model.predict_proba(X_one)[0]
    pred_class = int(proba.argmax())
    return Prediction(predicted_class=pred_class, probability=float(proba[pred_class]))

# Test with TestClient (the lifespan startup runs when entering the context).
with TestClient(serving_app) as client:
    print("\nGET /health:", client.get("/health").json())

    print("\nPOST /predict (valid):")
    r = client.post("/predict", json={"features": [0.5, -1.2, 0.3, 2.1]})
    print(f"  {r.json()}")

    print("\nPOST /predict (wrong feature count):")
    r = client.post("/predict", json={"features": [0.5, -1.2]})
    print(f"  status {r.status_code}: {r.json()['detail']}")


**The serving pattern to memorize** (everything above is the canonical shape of a model service):
1. **Load the model once at startup** via `lifespan` (not inside the endpoint — loading per request would be catastrophically slow).
2. **Validate input** with Pydantic + explicit checks (feature count, ranges).
3. **A `/health` endpoint** so load balancers and orchestrators (Kubernetes, ECS) know the service is alive and the model is loaded.
4. **Structured errors** via `HTTPException` with proper status codes (400 bad input, 503 not ready).
5. **Typed response** via `response_model`.

## 4.7 Running it for real (outside the notebook)

In a notebook we use `TestClient`. In production you save the app to a file and run a server:

```python
# main.py
from fastapi import FastAPI
app = FastAPI()

@app.get("/health")
def health():
    return {"status": "healthy"}
```

```bash
# Development: one process, auto-reload on code change
uvicorn main:app --reload --port 8000

# Production: multiple worker processes (one per core) for parallelism + throughput
uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4

# Or via gunicorn managing uvicorn workers (common production setup)
gunicorn main:app -w 4 -k uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000
```

**Why multiple workers:** one uvicorn worker = one process = one event loop on one core. Async gives you I/O concurrency *within* a worker, but to use **all your CPU cores** (and to get CPU parallelism for inference), you run **multiple worker processes**. Rule of thumb: `workers = (2 × cores) + 1` for I/O-heavy services, or `= cores` for CPU-heavy ones. This is multiprocessing (section 2) applied to web serving.

## 4.8 Interview Q&A — FastAPI

**Q: Why is FastAPI popular for ML serving?**
> Four reasons: (1) **async-native** (ASGI) — handles concurrent I/O-bound requests efficiently, which matters when your model service calls databases or other services. (2) **Pydantic validation** — request/response schemas from type hints, auto-rejecting bad input. (3) **automatic OpenAPI docs** at `/docs`. (4) **fast** — among the quickest Python frameworks. For putting a model behind an HTTP API, it's the 2026 default.

**Q: What's the difference between WSGI and ASGI?**
> WSGI is the older, **synchronous** Python web standard — one request per worker at a time, blocking on I/O (Flask, classic Django). ASGI is the **asynchronous** standard — a worker can interleave many concurrent requests at `await` points (FastAPI, Starlette). ASGI is better for I/O-bound services with many concurrent slow requests; WSGI is simpler and fine for fast CPU-bound or low-concurrency endpoints.

**Q (trap): Should every FastAPI endpoint be `async def`?**
> No. Use `async def` only when the endpoint does `await`-able async I/O. If your handler does **blocking** work (sync DB driver, CPU-bound `model.predict()`, a blocking library), write it as a plain `def` — FastAPI runs sync endpoints in a threadpool so they don't block the event loop. Putting blocking code in an `async def` freezes the loop and kills concurrency for all requests.

**Q: Where should you load your model — in the endpoint or at startup?**
> At startup, once, via the `lifespan` handler (or the older `@app.on_event("startup")`). Loading a model can take hundreds of ms to seconds; doing it per request would make every request unbearably slow and waste memory. Load once into a module-level holder, reference it in the endpoint.

**Q: Why run multiple uvicorn workers if FastAPI is async?**
> Async gives concurrency *within one process/core* (overlapping I/O waits). But one worker uses one core and one GIL. To use **all cores** — and to get real parallelism for CPU-bound inference — you run multiple worker processes (`--workers N`). Async handles per-worker I/O concurrency; multiple workers handle multi-core parallelism. You need both.

**Q (trap): My FastAPI model endpoint is `async def` and calls `model.predict()`, and the API gets slow under load. Why?**
> `model.predict()` is CPU-bound and **blocking** — inside an `async def` it hogs the event loop, blocking all other concurrent requests on that worker. Fixes: (a) make the endpoint a plain `def` so FastAPI threadpools it, (b) offload to a `ProcessPoolExecutor` for true parallelism, or (c) move inference to a dedicated serving layer. Also add more workers. The async-with-blocking-CPU combination is a classic footgun.


---
# 5. Model serving & inference

You have a model behind a FastAPI endpoint. Now: how do you serve it **efficiently** — low latency, high throughput, under real load? This section is concepts + the framework landscape, with one executable batching demo.

## 5.1 The two numbers that define serving: latency and throughput

- **Latency** — how long one request takes (end to end). Measured in percentiles: **p50** (median), **p95**, **p99** (tail). The tail matters most: "p99 latency = 800ms" means 1% of users wait 800ms+. SLAs are usually written on p95 or p99, not the average.
- **Throughput** — how many requests per second (RPS / QPS) the system handles.

These **trade off**. Batching requests together raises throughput (the GPU/CPU processes many at once efficiently) but raises latency (a request waits for the batch to fill). The serving system's job is to balance them for your SLA.

**Why percentiles, not averages:** averages hide tail behavior. A service with 50ms average latency might have 2s p99 — and that p99 is what frustrates users and trips timeouts. Always measure and report percentiles.

In [ ]:
# Demonstrate why you report percentiles, not the mean.
# Simulate latencies: most requests fast, a few slow (realistic tail).
np.random.seed(0)
fast = np.random.normal(50, 10, size=950)         # 950 requests ~50ms
slow = np.random.normal(500, 100, size=50)        # 50 requests ~500ms (the tail)
latencies = np.clip(np.concatenate([fast, slow]), 1, None)

print(f"Mean latency:   {latencies.mean():.0f} ms   <- looks fine")
print(f"p50 (median):   {np.percentile(latencies, 50):.0f} ms")
print(f"p95:            {np.percentile(latencies, 95):.0f} ms")
print(f"p99:            {np.percentile(latencies, 99):.0f} ms   <- the real user pain")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(latencies, bins=50, color="steelblue", edgecolor="black")
for p, c in [(50, "green"), (95, "orange"), (99, "red")]:
    v = np.percentile(latencies, p)
    ax.axvline(v, color=c, linestyle="--", label=f"p{p} = {v:.0f}ms")
ax.axvline(latencies.mean(), color="black", linestyle=":", label=f"mean = {latencies.mean():.0f}ms")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Count")
ax.set_title("Latency distribution — why the mean lies and p99 matters")
ax.legend()
plt.show()


## 5.2 Batching — the core throughput lever

Processing requests one at a time underuses hardware (especially GPUs, which are built for parallel batch math). **Batching** groups requests and runs inference on them together.

- **Static batching** — wait until you have N requests, then process. Simple but adds latency for the first request in a batch.
- **Dynamic batching** — process whatever has accumulated after a short time window (e.g. 5ms) OR when the batch hits max size, whichever first. Balances latency and throughput. This is what production serving frameworks (Triton, vLLM, TF Serving) do.

Demo: per-item vs batched inference on the same model.

In [ ]:
# Show batched inference is far more efficient than per-item.
model = joblib.load("/tmp/model.joblib")
X_load, _ = make_classification(n_samples=2000, n_features=4, n_informative=3,
                                 n_redundant=0, random_state=1)

# Per-item: call predict 2000 times (like handling 2000 separate requests one-by-one).
start = time.perf_counter()
per_item = [model.predict(x.reshape(1, -1))[0] for x in X_load]
per_item_time = time.perf_counter() - start

# Batched: one predict call on all 2000 rows.
start = time.perf_counter()
batched = model.predict(X_load)
batched_time = time.perf_counter() - start

print(f"Per-item (2000 separate calls): {per_item_time*1000:.0f} ms")
print(f"Batched   (1 call of 2000 rows): {batched_time*1000:.0f} ms")
print(f"Speedup from batching:           {per_item_time/batched_time:.0f}x")
print("\nThis is why serving frameworks batch: vectorized inference amortizes per-call overhead.")


## 5.3 The serving framework landscape

You don't always hand-roll serving with FastAPI. The options, from simplest to most specialized:

In [ ]:
serving_landscape = pd.DataFrame([
    ["FastAPI + uvicorn",  "Custom logic, single/few models, full control",
     "Simple, flexible, you own everything",  "You build batching/versioning/metrics yourself"],
    ["BentoML",            "Packaging + serving ML models with batching built in",
     "Adaptive batching, model store, easy deploy",  "Another framework to learn"],
    ["Ray Serve",          "Multi-model, Python-native, scalable pipelines",
     "Scales across a cluster, composable, autoscaling",  "Ray cluster overhead"],
    ["NVIDIA Triton",      "High-performance GPU serving, multi-framework",
     "Top GPU throughput, dynamic batching, multi-model",  "Heavier setup; mostly GPU"],
    ["TF Serving / TorchServe", "Framework-specific (TensorFlow / PyTorch) serving",
     "Optimized for their framework, versioning",  "Tied to one framework"],
    ["vLLM / TGI",         "LLM serving specifically",
     "PagedAttention, continuous batching, huge LLM throughput",  "LLM-only"],
], columns=["Tool", "Use for", "Strengths", "Tradeoffs"])
serving_landscape


**Decision guide:**
- **One sklearn/xgboost model, custom logic** → FastAPI + uvicorn. You've already built this.
- **Want batching/versioning without building it** → BentoML.
- **Many models, pipelines, cluster scale** → Ray Serve.
- **GPU models, max throughput** → Triton.
- **Serving LLMs** → vLLM or TGI (continuous batching is a game-changer for token generation).

**Don't over-engineer.** For most "serve one tabular model" jobs, FastAPI + uvicorn + a couple of workers is the right answer. Reach for the heavy tools when you hit real scale or GPU/LLM-specific needs.

## 5.4 The full serving stack — how the pieces fit

```
   Client request
        │
        ▼
   [Load balancer]          ── distributes across instances, health checks
        │
        ▼
   [uvicorn/gunicorn]       ── N worker processes (multiprocessing → use all cores)
        │
        ▼
   [FastAPI app]            ── async I/O concurrency per worker, validation
        │
        ▼
   [Model in memory]        ── loaded once at startup; optional dynamic batching
        │
        ▼
   [Response] + [metrics/logs/traces emitted]
```

Every layer maps to a concept in this notebook: load balancer (deployment, section 7), worker processes (multiprocessing, section 2), FastAPI async (sections 3-4), batching (this section), metrics/logs/traces (section 8).

---
# 6. Docker — packaging your service

**The problem Docker solves:** "it works on my machine." Your service depends on a specific Python version, specific library versions, system packages, environment variables. Docker packages *all of it* into an **image** that runs identically on your laptop, a colleague's machine, and the cloud.

**Docker can't run inside this notebook** (no Docker daemon in the sandbox), so this section writes real, correct files you copy to a project and explains them line by line. The commands are shown as text to run in a terminal.

## 6.1 Images vs containers — the core distinction

- An **image** is a **blueprint** — a frozen, layered filesystem with your code, dependencies, and runtime. Built once, stored, shared. (Like a class.)
- A **container** is a **running instance** of an image — an isolated process with its own filesystem view, network, etc. You can run many containers from one image. (Like an object.)

Lifecycle: write a `Dockerfile` → `docker build` produces an **image** → `docker run` starts a **container** from it → push the image to a **registry** (Docker Hub, AWS ECR) → the cloud pulls and runs it.

## 6.2 A Dockerfile for the FastAPI model service

Here's a production-shaped Dockerfile. We write it to disk so you can see the real file.

In [ ]:
dockerfile_content = '''# Start from a slim official Python base image (smaller = faster pulls, less attack surface).
FROM python:3.12-slim

# Set the working directory inside the container.
WORKDIR /app

# Copy ONLY requirements first — this is a layer-caching optimization (see notes below).
COPY requirements.txt .

# Install dependencies. --no-cache-dir keeps the image small.
RUN pip install --no-cache-dir -r requirements.txt

# Now copy the application code (changes often, so it comes AFTER deps).
COPY . .

# Document the port the app listens on.
EXPOSE 8000

# The command that runs when the container starts.
# 0.0.0.0 (not 127.0.0.1) so it accepts connections from outside the container.
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
'''

with open("/tmp/Dockerfile", "w") as f:
    f.write(dockerfile_content)
print("Wrote /tmp/Dockerfile:\n")
print(dockerfile_content)


## 6.3 The single most important Dockerfile optimization: layer caching

Each Dockerfile instruction creates a **layer**. Docker **caches** layers and only rebuilds from the first one that changed. This is why we `COPY requirements.txt` and `pip install` **before** `COPY . .`:

- Your **code** changes constantly; your **dependencies** rarely do.
- By installing deps first, Docker caches the (slow) `pip install` layer. When you change only code, Docker reuses the cached deps layer and just re-copies code — build takes seconds instead of minutes.
- If you copied everything first (`COPY . .` then `pip install`), *any* code change would invalidate the cache and force a full re-install every build.

**Order Dockerfile instructions from least-frequently-changed to most-frequently-changed.**

In [ ]:
# The companion requirements.txt and a multi-stage build example.
requirements = '''fastapi==0.136.1
uvicorn[standard]==0.47.0
scikit-learn==1.8.0
joblib==1.5.3
numpy==2.4.4
pydantic==2.13.4
'''
with open("/tmp/requirements.txt", "w") as f:
    f.write(requirements)

# Multi-stage build — compile/install in a "builder" stage, copy only what's needed
# into a slim final image. Keeps the final image small (no build tools shipped).
multistage = '''# ---- Stage 1: builder ----
FROM python:3.12-slim AS builder
WORKDIR /app
COPY requirements.txt .
# Install into a virtual env we can copy wholesale.
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"
RUN pip install --no-cache-dir -r requirements.txt

# ---- Stage 2: final (slim runtime) ----
FROM python:3.12-slim
# Copy ONLY the installed venv from the builder — no pip cache, no build tools.
COPY --from=builder /opt/venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"
WORKDIR /app
COPY . .
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
'''
with open("/tmp/Dockerfile.multistage", "w") as f:
    f.write(multistage)

print("Wrote requirements.txt and Dockerfile.multistage")
print("\nMulti-stage build:")
print(multistage)


## 6.4 The Docker commands (run in a terminal, not here)

```bash
# Build an image from the Dockerfile in the current directory, tag it.
docker build -t my-model-service:v1 .

# Run a container from the image, mapping container port 8000 to host port 8000.
docker run -p 8000:8000 my-model-service:v1

# Run in the background (detached), with a name and an env var.
docker run -d --name model-svc -p 8000:8000 -e LOG_LEVEL=info my-model-service:v1

# See running containers, view logs, stop it.
docker ps
docker logs model-svc
docker stop model-svc

# Push to a registry so the cloud can pull it.
docker tag my-model-service:v1 <account>.dkr.ecr.<region>.amazonaws.com/my-model-service:v1
docker push <account>.dkr.ecr.<region>.amazonaws.com/my-model-service:v1
```

## 6.5 `.dockerignore` and image-size hygiene

Add a `.dockerignore` (like `.gitignore`) so junk doesn't bloat your image or bust the cache:

```
__pycache__/
*.pyc
.git/
.venv/
notebooks/
data/
*.ipynb
.pytest_cache/
```

**Image-size best practices:**
- Use `-slim` base images (or `-alpine` for extreme minimalism, though it can cause C-extension headaches).
- Multi-stage builds to drop build tools from the final image.
- `--no-cache-dir` on pip.
- Combine `RUN` commands with `&&` to reduce layers.
- Don't copy data/models you can mount or download at runtime (unless you specifically want them baked in).

## 6.6 Interview Q&A — Docker

**Q: What's the difference between a Docker image and a container?**
> An image is the immutable blueprint — a layered filesystem snapshot of your code + dependencies + runtime. A container is a running instance of an image — an isolated process. One image, many containers. Analogy: image is a class, container is an object.

**Q: Why does Dockerfile instruction order matter?**
> Layer caching. Each instruction is a cached layer; Docker rebuilds from the first changed layer onward. Put rarely-changing steps (installing dependencies) before frequently-changing ones (copying code). That way a code change reuses the cached dependency layer and rebuilds in seconds instead of re-running `pip install` every time.

**Q: Why `--host 0.0.0.0` and not `127.0.0.1` in the container's CMD?**
> `127.0.0.1` only accepts connections from inside the container itself, so port mapping wouldn't reach your app. `0.0.0.0` binds all interfaces, letting traffic from outside the container (via `docker run -p`) reach the server. Forgetting this is a classic "why can't I connect to my container" bug.

**Q: What's a multi-stage build and why use it?**
> A Dockerfile with multiple `FROM` stages. You build/install in an early "builder" stage (with compilers, build tools), then copy only the finished artifacts into a clean slim final stage. The final image excludes build tools and caches → much smaller, faster to pull, smaller attack surface. Standard for production images.

**Q (trap): My Docker image is 3GB. How do I shrink it?**
> Several levers: use a `-slim` base instead of full `python` (saves ~700MB), multi-stage build to drop build tools, `pip install --no-cache-dir`, a `.dockerignore` to exclude data/notebooks/git, and don't bake large datasets/models into the image if you can mount or download them at runtime. Also combine `RUN` layers and clean apt caches in the same layer.


---
# 7. Cloud deployment

Your service is in a Docker image. Where does it actually run, and how does it scale? This is conceptual + a decision framework (you can't deploy from a notebook), oriented around the choices you'll actually face.

## 7.1 The four ways to run your service in the cloud

From most control / most ops work, to least:

| Option | What it is | You manage | Cloud manages | Example |
|---|---|---|---|---|
| **VM** | A rented virtual machine | OS, runtime, scaling, your app | Hardware | AWS EC2, GCP Compute Engine |
| **Container service** | Run containers, cloud handles the hosts | Container image, config | Hosts, orchestration, scaling | AWS ECS/Fargate, GCP Cloud Run, K8s |
| **Serverless** | Run a function per request, no servers | Just the function code | Everything else (scales to zero) | AWS Lambda, GCP Cloud Functions |
| **Managed ML** | Purpose-built ML serving | Model artifact + config | Serving infra, autoscaling, monitoring | AWS SageMaker, GCP Vertex AI |

The trend over time: teams move **down** this list (less ops) as their needs allow. You start on EC2 because it's familiar, then realize Cloud Run / Fargate removes the server-management toil.

## 7.2 When to use which

In [ ]:
cloud_choice = pd.DataFrame([
    ["VM (EC2)",            "Full control, special hardware/drivers, legacy apps, persistent state",
     "You manage OS patches, scaling, deployment — most ops burden"],
    ["Container (Fargate/Cloud Run)", "Most stateless model APIs — the sweet spot in 2026",
     "Push an image, it runs and autoscales; no server management"],
    ["Serverless (Lambda)", "Spiky/infrequent traffic, event-driven, tiny models, scale-to-zero",
     "Cold starts, size/time limits, awkward for big ML deps & GPUs"],
    ["Managed ML (SageMaker/Vertex)", "Want batteries-included: autoscaling, A/B, monitoring, GPU",
     "Vendor lock-in, cost, less flexibility, learning curve"],
], columns=["Option", "Best for", "Cost / caveat"])
cloud_choice


**The 2026 default for a stateless model API:** a **container service** like AWS Fargate or GCP Cloud Run. You push your Docker image, set CPU/memory and min/max instances, and it autoscales on traffic — no servers to patch. Reach for **VMs** when you need specific GPUs/drivers or persistent state, **serverless** for spiky low-volume or event-driven workloads (mind cold starts and the ~250MB-ish dependency limits that bite ML), and **managed ML platforms** when you want built-in autoscaling/monitoring/A-B testing and accept the lock-in.

## 7.3 The deploy path — image to running service

```
   [Your code + Dockerfile]
        │  docker build
        ▼
   [Docker image]
        │  docker push
        ▼
   [Container registry]      ── AWS ECR, Docker Hub, GCP Artifact Registry
        │  deploy (console / CLI / CI-CD / Terraform)
        ▼
   [Container service]       ── Fargate / Cloud Run / Kubernetes
        │  pulls image, runs N replicas behind a load balancer
        ▼
   [Live service]  ──  https://api.yourcompany.com/predict
```

In practice this is automated by **CI/CD**: a push to your main branch triggers a pipeline (GitHub Actions, GitLab CI) that builds the image, pushes to the registry, and tells the container service to roll out the new version.

## 7.4 Autoscaling and the concepts that matter

- **Horizontal scaling** (scale *out*): add more instances/replicas. The cloud-native way — stateless services scale horizontally behind a load balancer. This is why you keep services **stateless** (no in-memory session state; put state in a database/cache).
- **Vertical scaling** (scale *up*): bigger instance (more CPU/RAM/GPU). Limited ceiling; needed for big models that must fit in memory.
- **Autoscaling policy:** add replicas when a metric (CPU %, request count, p95 latency) crosses a threshold; remove them when it drops. Set **min replicas** (avoid cold starts / keep a model warm) and **max replicas** (cap cost).
- **Load balancer:** distributes incoming requests across replicas and uses your `/health` endpoint to route only to healthy ones.

**Why statelessness matters:** if any replica can handle any request (no per-instance memory of the user), you can add/remove replicas freely and the load balancer can route anywhere. State lives in external stores (Postgres, Redis, S3), not in the process.

## 7.5 Interview Q&A — deployment

**Q: EC2 vs Fargate vs Lambda for a model API — how do you choose?**
> EC2 (a VM) gives full control but you manage the OS, scaling, and deploys — most ops burden; pick it for special hardware/GPU/driver needs or persistent state. Fargate (containers) is the sweet spot for stateless model APIs — push an image, it autoscales, no server management. Lambda (serverless) suits spiky/infrequent or event-driven traffic and scales to zero, but cold starts and dependency-size limits make it awkward for large ML deps and GPUs. For a typical always-on tabular model API: Fargate or Cloud Run.

**Q: What's the difference between horizontal and vertical scaling?**
> Horizontal = more instances (scale out); vertical = a bigger instance (scale up). Cloud-native services scale horizontally behind a load balancer because it has no ceiling and no single point of failure — which requires the service to be stateless. Vertical scaling is needed when a single request/model must fit in more memory or use a bigger GPU, but it has a hard upper limit.

**Q: Why should a serving service be stateless?**
> So any replica can handle any request, letting you add/remove replicas freely and route traffic anywhere via the load balancer. If a replica held per-user state in memory, requests would have to stick to specific instances, breaking autoscaling and creating data loss when an instance dies. Keep state in external stores (DB, Redis, S3); keep the service itself stateless.

**Q: What's the role of the `/health` endpoint in deployment?**
> Load balancers and orchestrators (ECS, Kubernetes) poll it to decide whether a replica is ready to receive traffic and whether to restart it. A good health check confirms the service is up *and* the model is loaded (not just that the process is alive). Returning unhealthy during model load prevents traffic from hitting a not-ready replica.

**Q (trap): Why might serverless (Lambda) be a bad fit for ML serving?**
> Three reasons: (1) **cold starts** — when a function scales from zero, the first request waits for the runtime + model to load, which can be seconds for ML. (2) **package size limits** — ML dependencies (torch, etc.) often blow past serverless size limits. (3) **no GPU / limited compute + time limits**. Serverless shines for lightweight, spiky, event-driven work; for always-on or heavy ML it's usually a container service instead.


---
# 8. Observability — logging, metrics, tracing

Your service is live. How do you know it's working, and how do you debug it when it isn't? **Observability** = the ability to understand a system's internal state from its outputs. It has **three pillars**, and you specifically asked for the refined difference between them.

## 8.1 The three pillars — the crisp distinction

| Pillar | Question it answers | Granularity | Example |
|---|---|---|---|
| **Logs** | "What happened (in detail) at this moment?" | Per-event, high detail | "2026-05-20 10:32:01 ERROR prediction failed: feature count mismatch, request_id=abc123" |
| **Metrics** | "What's the aggregate health over time?" | Aggregated numbers over time | "p99 latency = 240ms; 1500 req/s; 0.3% error rate" |
| **Traces** | "Where did the time go across services for this one request?" | Per-request, across services | "request abc123: 5ms in API → 180ms in model → 12ms in DB" |

The mental model:
- **Logs** are the detailed diary — rich, per-event, great for debugging a specific incident, but expensive to store and hard to aggregate.
- **Metrics** are the dashboard gauges — cheap, aggregated, great for alerting and trends, but no per-event detail.
- **Traces** are the request's journey — they follow ONE request as it hops across services, showing where latency accumulates. Essential in microservices.

**They're complementary, not redundant.** Metrics tell you *something is wrong* (error rate spiked). Traces tell you *where* (the model service, not the database). Logs tell you *what exactly* (the specific exception and inputs).

## 8.2 Logging — structured logs in Python (executable)

The Python `logging` module is the standard. The key upgrade for production: **structured logging** (JSON), so machines can parse and query your logs, not just humans reading text.

In [ ]:
import logging
import json

# Basic logging with levels.
logger = logging.getLogger("model_service")
logger.setLevel(logging.INFO)
# Avoid duplicate handlers if the cell runs twice.
logger.handlers.clear()
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s [%(name)s] %(message)s"))
logger.addHandler(handler)

# The five standard levels, in increasing severity.
logger.debug("Detailed diagnostic info (usually off in production)")
logger.info("Normal operational event — request served")
logger.warning("Something unexpected but handled — falling back to default model")
logger.error("A request failed — but the service is still up")
logger.critical("The service itself is broken — model failed to load")


In [ ]:
# Structured logging: emit JSON so log aggregators (CloudWatch, Datadog, ELK) can query fields.
class JSONFormatter(logging.Formatter):
    def format(self, record):
        log_obj = {
            "timestamp": self.formatTime(record),
            "level": record.levelname,
            "service": record.name,
            "message": record.getMessage(),
        }
        # Attach any extra structured fields.
        if hasattr(record, "extra_fields"):
            log_obj.update(record.extra_fields)
        return json.dumps(log_obj)

structured_logger = logging.getLogger("structured_service")
structured_logger.setLevel(logging.INFO)
structured_logger.handlers.clear()
h = logging.StreamHandler()
h.setFormatter(JSONFormatter())
structured_logger.addHandler(h)

# Log a prediction event with structured fields you can later query/aggregate.
def log_prediction(request_id, latency_ms, predicted_class, confidence):
    structured_logger.info(
        "prediction served",
        extra={"extra_fields": {
            "request_id": request_id,
            "latency_ms": latency_ms,
            "predicted_class": predicted_class,
            "confidence": confidence,
        }},
    )

log_prediction("req-abc123", 45.2, 1, 0.93)
log_prediction("req-def456", 38.1, 0, 0.88)
print("\nThese JSON lines can be queried like: 'show all predictions where latency_ms > 100'")


**Why structured (JSON) logs:** in production your logs flow into an aggregator (CloudWatch Logs, Datadog, Elasticsearch). With JSON, you can query `latency_ms > 100 AND predicted_class = 1` across millions of log lines. With plain-text logs you're stuck grepping. Always attach a **`request_id`** (a.k.a. correlation ID) so you can trace one request's logs across services.

**Logging best practices:**
- Log at the right level (INFO for normal events, ERROR for failures, DEBUG off in prod).
- Never log secrets, PII, or full feature vectors (privacy + size).
- Include a `request_id` on every log line for correlation.
- Log decisions and failures, not every line of code.

## 8.3 Metrics — what to measure (the RED and USE methods)

Metrics are aggregated numbers tracked over time, scraped by systems like **Prometheus** and visualized in **Grafana**. Two standard frameworks for *what* to measure:

**RED method** (for request-driven services — your model API):
- **R**ate — requests per second
- **E**rrors — failed requests per second (or error rate %)
- **D**uration — latency distribution (p50/p95/p99)

**USE method** (for resources — the machine):
- **U**tilization — % busy (CPU, GPU, memory)
- **S**aturation — how much work is queued/waiting
- **E**rrors — hardware/resource errors

For an ML service you track the RED metrics plus resource USE metrics. Demo: a tiny in-memory metrics tracker like what a real metrics client does under the hood.

In [ ]:
from collections import defaultdict

class SimpleMetrics:
    '''A toy metrics collector — illustrates what Prometheus client libs do internally.'''
    def __init__(self):
        self.counters = defaultdict(int)
        self.latencies = []

    def increment(self, name, value=1):
        self.counters[name] += value

    def observe_latency(self, ms):
        self.latencies.append(ms)

    def report(self):
        lat = np.array(self.latencies) if self.latencies else np.array([0])
        return {
            "total_requests": self.counters["requests"],
            "total_errors": self.counters["errors"],
            "error_rate": self.counters["errors"] / max(self.counters["requests"], 1),
            "p50_ms": float(np.percentile(lat, 50)),
            "p95_ms": float(np.percentile(lat, 95)),
            "p99_ms": float(np.percentile(lat, 99)),
        }

# Simulate serving 1000 requests with realistic latencies and a few errors.
metrics = SimpleMetrics()
np.random.seed(0)
for _ in range(1000):
    metrics.increment("requests")
    latency = np.random.gamma(2, 20)            # realistic right-skewed latency
    metrics.observe_latency(latency)
    if np.random.random() < 0.02:               # 2% error rate
        metrics.increment("errors")

report = metrics.report()
print("RED metrics dashboard:")
for k, v in report.items():
    print(f"  {k:<16s}: {v:.3f}" if isinstance(v, float) else f"  {k:<16s}: {v}")


**In production you don't hand-roll this** — you use `prometheus-client` (exposes a `/metrics` endpoint that Prometheus scrapes) or a vendor SDK (Datadog, CloudWatch). The point of the demo is to show metrics are just **counters and distributions aggregated over time** — rate, errors, duration. You set **alerts** on them, e.g. page me if error rate exceeds 1% for 5 minutes, or if p99 latency goes above 500ms.

## 8.4 Tracing — following one request across services

In a microservices system, one user request might hop through: API gateway → auth service → model service → feature store → database. If it's slow, **which hop** is to blame? **Distributed tracing** answers this.

A **trace** follows one request end-to-end. It's made of **spans** — each span is one unit of work (one service call, one DB query) with a start time, duration, and parent span. Together they form a tree showing exactly where time went.

The standard is **OpenTelemetry** (vendor-neutral); backends include Jaeger, Zipkin, Datadog APM. Demo: a simplified span tree.

In [ ]:
import contextlib

class SimpleTracer:
    '''Toy tracer illustrating spans — real tracing uses OpenTelemetry.'''
    def __init__(self):
        self.spans = []
        self.depth = 0

    @contextlib.contextmanager
    def span(self, name):
        start = time.perf_counter()
        depth = self.depth
        self.depth += 1
        try:
            yield
        finally:
            self.depth -= 1
            duration_ms = (time.perf_counter() - start) * 1000
            self.spans.append((depth, name, duration_ms))

    def print_trace(self):
        print("Trace for request req-abc123:")
        for depth, name, dur in self.spans:
            print(f"  {'  ' * depth}{'└─' if depth else ''}{name}: {dur:.1f}ms")

# Simulate a request flowing through services, each a span.
tracer = SimpleTracer()
with tracer.span("POST /predict (total)"):
    with tracer.span("auth check"):
        time.sleep(0.005)
    with tracer.span("fetch features from store"):
        time.sleep(0.030)
    with tracer.span("model inference"):
        time.sleep(0.080)
    with tracer.span("write prediction log"):
        time.sleep(0.010)

tracer.print_trace()
print("\nThe trace shows model inference (80ms) dominates — that's where to optimize.")


**Why tracing matters and metrics/logs don't replace it:** metrics tell you p99 latency spiked to 800ms. But *where*? The trace shows the request spent 700ms in the feature store, not the model. Without tracing you'd be guessing. In a monolith you can often live without it; in microservices it's essential.

## 8.5 ML-specific monitoring — the part generic observability misses

A model can be perfectly healthy by infra metrics (low latency, no errors) while **silently making worse predictions** because the world changed. This is unique to ML and easy to forget.

- **Data drift** — the distribution of incoming features shifts away from training data (e.g., a new user demographic, a changed upstream feature). Detect with statistical tests (KS test, population stability index) comparing live feature distributions to a training reference.
- **Prediction drift** — the distribution of the model's outputs shifts (e.g., suddenly predicting "fraud" 3x more often). A cheap early-warning signal even without ground truth.
- **Concept drift** — the relationship between features and target changes (the model's learned mapping is now wrong). Detectable only once you get ground-truth labels back.
- **Model performance decay** — accuracy/AUC computed on delayed ground truth, tracked over time. The ultimate signal, but lags because labels arrive late.

Demo: a simple data-drift check with the KS test.

In [ ]:
from scipy import stats

# Reference: feature distribution at training time.
np.random.seed(0)
training_feature = np.random.normal(50, 10, size=5000)

# Two production scenarios.
prod_no_drift = np.random.normal(50, 10, size=1000)       # same distribution
prod_with_drift = np.random.normal(58, 12, size=1000)     # shifted — drift!

for name, prod in [("no drift", prod_no_drift), ("with drift", prod_with_drift)]:
    ks_stat, p_value = stats.ks_2samp(training_feature, prod)
    drifted = p_value < 0.05
    print(f"{name:<12s}: KS stat={ks_stat:.3f}, p-value={p_value:.4f} -> "
          f"{'DRIFT DETECTED — investigate/retrain' if drifted else 'no significant drift'}")

# Visualize.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (name, prod) in zip(axes, [("no drift", prod_no_drift), ("with drift", prod_with_drift)]):
    ax.hist(training_feature, bins=40, alpha=0.5, density=True, label="training (reference)", color="steelblue")
    ax.hist(prod, bins=40, alpha=0.5, density=True, label=f"production ({name})", color="tomato")
    ax.set_title(f"Feature distribution — {name}")
    ax.legend()
plt.tight_layout()
plt.show()


**The production monitoring stack for ML** therefore has two layers:
1. **Infra observability** (logs, metrics, traces) — is the *service* healthy? Latency, errors, throughput.
2. **ML observability** (drift, performance decay) — is the *model* still good? Data drift, prediction drift, accuracy on delayed labels.

Tools like Evidently, WhyLabs, Arize, and Fiddler specialize in layer 2. A model that passes every infra check can still be quietly failing — which is why ML monitoring is its own discipline.

## 8.6 Interview Q&A — observability

**Q: What's the difference between logging, metrics, and tracing?**
> **Logs** are detailed per-event records ("what happened at this moment") — rich, great for debugging a specific incident, expensive to store. **Metrics** are aggregated numbers over time ("what's the overall health") — cheap, great for dashboards and alerts, no per-event detail. **Traces** follow one request across services ("where did the time go for this request") — essential for finding latency bottlenecks in microservices. They're complementary: metrics say *something's wrong*, traces say *where*, logs say *what exactly*.

**Q: What metrics would you track for a model-serving API?**
> The RED method: **R**ate (requests/sec), **E**rrors (error rate), **D**uration (latency p50/p95/p99). Plus resource USE metrics (CPU/GPU/memory utilization and saturation). And — uniquely for ML — **drift and model-performance** metrics, because a service can be infra-healthy while the model silently degrades.

**Q (trap): My model service has great latency and zero errors, but business metrics are dropping. What might be wrong?**
> The model is probably **drifting** — infra is healthy but predictions have degraded because the input distribution or the feature-target relationship changed since training. Generic observability (logs/metrics/traces) won't catch this; you need ML-specific monitoring: data drift (KS test / PSI on feature distributions), prediction drift (output distribution shift), and model performance on delayed ground-truth labels. The fix is usually retraining on recent data.

**Q: Why report p99 latency instead of average?**
> Averages hide the tail. A service with 50ms average can have 2s p99 — meaning 1% of requests are painfully slow, tripping client timeouts and frustrating users. SLAs are written on percentiles (p95/p99) precisely because tail latency is what users feel and what cascades into failures in distributed systems.

**Q: What is distributed tracing and when do you need it?**
> Tracing follows a single request as it flows across multiple services, recording a tree of timed **spans** (one per unit of work). You need it in **microservice** architectures, where a slow request could be blamed on any of several hops — the trace pinpoints which. In a single monolithic service, logs and metrics often suffice; the value of tracing grows with the number of services a request touches.

**Q: What's a correlation ID / request ID and why does it matter?**
> A unique ID attached to a request and propagated through every service and log line it touches. It lets you reconstruct everything that happened for one specific request across distributed logs and link logs to traces. Without it, debugging "what happened to request X" in a high-traffic multi-service system is nearly impossible. Generate it at the entry point (or accept it from the client) and pass it downstream.


---
# Closeout

## What you covered

The path from `model.predict()` to a production service:

1. **Concurrency** — I/O-bound vs CPU-bound is the master distinction. Threading and asyncio for I/O concurrency; multiprocessing for CPU parallelism (the GIL is why). 
2. **Asyncio** — the event loop is one thread that parks waiting tasks and runs ready ones. `await` yields control; blocking calls freeze the loop. `gather` fans out concurrent I/O.
3. **FastAPI** — async-native (ASGI) web framework; Pydantic validation; `async def` for awaitable I/O, plain `def` for blocking work; load the model once at startup.
4. **Serving** — latency (p99!) vs throughput; batching is the throughput lever; pick FastAPI for custom serving, dedicated frameworks (Triton/vLLM/BentoML) for scale.
5. **Docker** — images vs containers; order Dockerfile layers least-to-most-changing for caching; multi-stage builds for small images.
6. **Cloud** — VM vs container vs serverless vs managed; containers (Fargate/Cloud Run) are the default for stateless model APIs; scale horizontally, stay stateless.
7. **Observability** — logs (per-event detail), metrics (aggregate health, RED method), traces (per-request journey); plus ML-specific drift monitoring.

## The mental model, one more time

```
   CONCURRENCY (the foundation: handle many requests without single-file waiting)
        ▼
   FASTAPI (requests reach your model over HTTP, validated)
        ▼
   SERVING (inference is efficient: batching, workers, low p99)
        ▼
   DOCKER (packaged to run identically everywhere)
        ▼
   CLOUD (runs and autoscales)
        ▼
   OBSERVABILITY (you know it works, and why when it doesn't)
```

## The interview synthesis

When asked **"how would you deploy this model?"**, a strong answer walks the stack:
> "I'd wrap the model in a FastAPI service — load the model once at startup, validate inputs with Pydantic, expose `/predict` and `/health`. For an I/O-light tabular model I'd use sync endpoints in a threadpool with a few uvicorn workers to use all cores. I'd containerize with a slim multi-stage Dockerfile, push to a registry, and deploy on a container service like Fargate or Cloud Run with autoscaling on p95 latency, keeping the service stateless. For observability I'd emit structured JSON logs with request IDs, track RED metrics with alerts on error rate and p99, and add ML-specific drift monitoring since infra health doesn't catch model decay."

That single paragraph touches every section of this notebook.

## What's next

You now have four masterclasses: supervised ML, unsupervised ML, evaluation metrics, and productionization. Natural next steps:
- **ML System Design** — the architecture-level view: recommendation funnels, feature stores, training/serving skew, online vs offline, the full design-interview framework.
- **A/B testing & experimentation** — going from offline metrics to online metrics that prove business impact.
- **A drill/quiz session** on any of these four notebooks.

Tell me which.
